In [15]:
#Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import graphviz
from sklearn.metrics import make_scorer, roc_auc_score


from sklearn.model_selection import train_test_split,GridSearchCV,RepeatedStratifiedKFold
from sklearn import metrics
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, export_graphviz 

In [16]:
train_data = pd.read_csv("../data/CENSUS_ED_ATTN.csv")
test_data = pd.read_csv("../data/Census_Test.csv")

In [17]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151747 entries, 0 to 151746
Data columns (total 15 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   A_MARITL   151747 non-null  int64
 1   A_SEX      151747 non-null  int64
 2   PEAFEVER   151747 non-null  int64
 3   PARENT     151747 non-null  int64
 4   PENATVTY   151747 non-null  int64
 5   PEFNTVTY   151747 non-null  int64
 6   PEHSPNON   151747 non-null  int64
 7   PEINUSYR   151747 non-null  int64
 8   PEPAR1TYP  151747 non-null  int64
 9   PRCITSHP   151747 non-null  int64
 10  PRDTRACE   151747 non-null  int64
 11  ERN_SRCE   151747 non-null  int64
 12  WSAL_VAL   151747 non-null  int64
 13  ANN_VAL    151747 non-null  int64
 14  A_HGA      151747 non-null  int64
dtypes: int64(15)
memory usage: 17.4 MB


In [18]:
train_data.isna().sum()

A_MARITL     0
A_SEX        0
PEAFEVER     0
PARENT       0
PENATVTY     0
PEFNTVTY     0
PEHSPNON     0
PEINUSYR     0
PEPAR1TYP    0
PRCITSHP     0
PRDTRACE     0
ERN_SRCE     0
WSAL_VAL     0
ANN_VAL      0
A_HGA        0
dtype: int64

In [19]:
print(train_data['A_HGA'].value_counts())

A_HGA
39    33254
0     31453
43    24260
40    18624
44    10604
42     6457
41     4873
37     4189
36     3691
35     3273
34     2610
46     2263
38     2122
45     1575
33     1410
32      746
31      343
Name: count, dtype: int64


In [20]:
print(train_data['PENATVTY'].value_counts())

PENATVTY
57     128740
303      6116
210      1298
207      1070
233      1041
        ...  
105         3
523         3
149         3
425         2
155         2
Name: count, Length: 161, dtype: int64


In [21]:
# Dropping military service
train_data = train_data.drop('PEAFEVER', axis=1)
train_data = train_data.drop('PEHSPNON', axis=1)

In [22]:
# Repeating for test data
test_data = test_data.drop('PEAFEVER', axis=1)
test_data = test_data.drop('PEHSPNON', axis=1)

In [23]:
# Collapsing country to immigrant vs. from U.S
train_data['PENATVTY'] = train_data['PENATVTY'].apply(lambda x: 0 if x == 57 else 1)
train_data['PEFNTVTY'] = train_data['PEFNTVTY'].apply(lambda x: 0 if x == 57 else 1)


In [24]:
# Repeating for test data
test_data['PENATVTY'] = test_data['PENATVTY'].apply(lambda x: 0 if x == 57 else 1)
test_data['PEFNTVTY'] = test_data['PEFNTVTY'].apply(lambda x: 0 if x == 57 else 1)

In [25]:
train_data['PRDTRACE'] = train_data['PRDTRACE'].apply(lambda x: 0 if x == 1 else 1)

In [26]:
test_data['PRDTRACE'] = test_data['PRDTRACE'].apply(lambda x: 0 if x == 1 else 1)

In [27]:
ordinal_list = ['A_SEX', 'PARENT', 'PENATVTY', 'PEFNTVTY', 'PEINUSYR', 'PEPAR1TYP', 'PRCITSHP', 'PRDTRACE', 'ERN_SRCE']
ordinal_encoder = OrdinalEncoder()
for i in ordinal_list:
    train_data[[i]] = ordinal_encoder.fit_transform(train_data[[i]])

In [28]:
ordinal_list = ['A_SEX', 'PARENT', 'PENATVTY', 'PEFNTVTY', 'PEINUSYR', 'PEPAR1TYP', 'PRCITSHP', 'PRDTRACE', 'ERN_SRCE']
ordinal_encoder = OrdinalEncoder()
for i in ordinal_list:
    test_data[[i]] = ordinal_encoder.fit_transform(test_data[[i]])

X= train_data.drop(columns='A_HGA')

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.70, stratify = y, random_state=21)
X_tune, X_test, y_tune, y_test = train_test_split(X_test,y_test,  train_size = 0.50, stratify = y_test, random_state=49)

kf = RepeatedStratifiedKFold(n_splits=10,n_repeats =5, random_state=42)

param = {
    'max_depth': [2, 4, 6, 8, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

cl= DecisionTreeClassifier(random_state=672)

scoring = ['roc_auc_ovr', 'precision_macro', 'balanced_accuracy']

search = GridSearchCV(cl, param, scoring=scoring, n_jobs=-1, cv=kf,refit='precision_macro')

model = search.fit(X_train, y_train)

In [79]:
from sklearn.metrics import make_scorer, precision_score


target_encoder = OrdinalEncoder()
y = target_encoder.fit_transform(train_data[['A_HGA']]).ravel()
X = train_data.drop(columns='A_HGA')

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.70, stratify = y, random_state=21)
X_tune, X_test, y_tune, y_test = train_test_split(X_test,y_test,  train_size = 0.50, stratify = y_test, random_state=49)

scoring = ['roc_auc_ovr', 'balanced_accuracy', 'recall']

kf = RepeatedStratifiedKFold(n_splits=8,n_repeats =4, random_state=482)

param = param = {'max_depth': [8],
    'min_samples_split': [5],
    'min_samples_leaf': [3],
    'criterion': ['entropy']}

cl = DecisionTreeClassifier(random_state=462, class_weight='balanced')

search = GridSearchCV(cl, param, scoring=scoring, cv=kf,n_jobs=-1, refit = 'balanced_accuracy')
model = search.fit(X_train, y_train)

/home/vscode/.local/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:978: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 140, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 388, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.12/site-packages/sklearn/utils/_param_validation.py", line 216, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py", line 2429, in recall_score
    _, r, _, _ = precision_recall

In [81]:
from sklearn.metrics import make_scorer, precision_score

target_encoder = OrdinalEncoder()
y = target_encoder.fit_transform(train_data[['A_HGA']]).ravel()
X = train_data.drop(columns='A_HGA')

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.70, stratify=y, random_state=21)
X_tune, X_test, y_tune, y_test = train_test_split(X_test, y_test, train_size=0.50, stratify=y_test, random_state=49)

scoring = ['precision_macro', 'balanced_accuracy', 'recall']

kf = RepeatedStratifiedKFold(n_splits=8, n_repeats=4, random_state=482)

param = {
    'max_depth': [8],
    'min_samples_split': [5],
    'min_samples_leaf': [3],
    'criterion': ['entropy']
}

cl = DecisionTreeClassifier(random_state=462, class_weight='balanced')

search = GridSearchCV(cl, param, scoring=scoring, cv=kf, n_jobs=-1, refit='precision_macro')
model = search.fit(X_train, y_train)

/home/vscode/.local/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:978: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 140, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 388, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.12/site-packages/sklearn/utils/_param_validation.py", line 216, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py", line 2429, in recall_score
    _, r, _, _ = precision_recall

1. param = {'max_depth': [4, 8, 14, 16],
    'min_samples_split': [3, 5, 7],
    'min_samples_leaf': [3, 5, 7]}
2. best = model.best_estimator_
print(best)
test_predictions = best.predict(test_data)
original_predictions = target_encoder.inverse_transform(test_predictions.reshape(-1, 1))
predictions = pd.DataFrame({'id': test_data.index+1, 'A_HGA': original_predictions.ravel()})

In [83]:
best = model.best_estimator_
y_pred = best.predict(X_test)
macro_precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
print(f"Macro Precision: {macro_precision:.3f}")

Macro Precision: 0.193


In [78]:
best

DecisionTreeClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=8, min_samples_leaf=3, min_samples_split=5,
                       random_state=462)

In [84]:
best = model.best_estimator_
print(best)
test_predictions = best.predict(test_data)
original_predictions = target_encoder.inverse_transform(test_predictions.reshape(-1, 1))
predictions = pd.DataFrame({'id': test_data.index+1, 'A_HGA': original_predictions.ravel()})

DecisionTreeClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=8, min_samples_leaf=3, min_samples_split=5,
                       random_state=462)


In [ ]:
y_pred = best.predict(X_test)
macro_precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
print(f"Macro Precision: {macro_precision:.3f}")

Macro Precision: 0.160


In [ ]:
test_predictions = best.predict(test_data)
original_predictions = target_encoder.inverse_transform(test_predictions.reshape(-1, 1))
predictions = pd.DataFrame({'id': test_data.index, 'prediction': original_predictions.ravel()})

### Testing a KNN model

In [ ]:
target_encoder = OrdinalEncoder()
y = target_encoder.fit_transform([['A_HGA']]).ravel()
X = train_data.drop(columns='A_HGA')

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.70, stratify = y, random_state=21)
X_tune, X_test, y_tune, y_test = train_test_split(X_test,y_test,  train_size = 0.50, stratify = y_test, random_state=49)

In [36]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import make_scorer, precision_score


# Initialize the KNN model
knn = KNeighborsClassifier(n_neighbors=5)

# Fit the model
knn.fit(X_train, y_train)

# Predict on the test set
y_pred_knn = knn.predict(test_data)

# Evaluate the model

#macro_precision = precision_score(y_test, y_pred_knn, average='macro', zero_division=0)
#print(f"Macro Precision: {macro_precision:.3f}")
predictions = target_encoder.inverse_transform(y_pred_knn.reshape(-1, 1))
predictions_df = pd.DataFrame({'id': test_data.index+1, 'A_HGA': y_pred_knn.astype(int).ravel()})

In [35]:
csv = predictions_df.to_csv('predictions.csv', index=False)